# Split Criteria: ID3, C4.5, and CART Mechanics

This notebook computes common split criteria directly: entropy, information gain, gain ratio, Gini impurity, and threshold search for continuous attributes. The goal is to inspect the algorithmic mechanics behind classical tree induction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(17)

In [ ]:
n = 240
outlook = rng.choice(["sunny", "overcast", "rain"], size=n, p=[0.42, 0.24, 0.34])
humidity = rng.normal(62, 18, size=n).clip(20, 100)
wind = rng.normal(12, 5, size=n).clip(0, 35)
temperature = rng.normal(72, 11, size=n).clip(35, 105)

score = (
    (outlook == "overcast") * 1.2
    + (outlook == "rain") * 0.35
    - (outlook == "sunny") * 0.45
    - 0.035 * (humidity - 60)
    - 0.045 * np.maximum(wind - 14, 0)
    + 0.020 * (temperature - 70)
    + rng.normal(0, 0.45, size=n)
)
y = (score > 0).astype(int)

data = pd.DataFrame({
    "outlook": outlook,
    "humidity": humidity,
    "wind": wind,
    "temperature": temperature,
    "play": y,
})
data.head()

In [ ]:
def class_counts(labels):
    values, counts = np.unique(labels, return_counts=True)
    return dict(zip(values, counts))

def entropy(labels):
    counts = np.array(list(class_counts(labels).values()), dtype=float)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p))

def gini(labels):
    counts = np.array(list(class_counts(labels).values()), dtype=float)
    p = counts / counts.sum()
    return 1 - np.sum(p ** 2)

def weighted_impurity(groups, impurity_fn):
    total = sum(len(g) for g in groups)
    return sum((len(g) / total) * impurity_fn(g) for g in groups if len(g))

print("Parent entropy:", entropy(data["play"]))
print("Parent Gini:", gini(data["play"]))

In [ ]:
def categorical_split_score(frame, feature, target="play"):
    groups = [group[target].to_numpy() for _, group in frame.groupby(feature)]
    parent_entropy = entropy(frame[target].to_numpy())
    parent_gini = gini(frame[target].to_numpy())
    info_gain = parent_entropy - weighted_impurity(groups, entropy)
    gini_gain = parent_gini - weighted_impurity(groups, gini)
    split_info = entropy(frame[feature].to_numpy())
    gain_ratio = info_gain / split_info if split_info > 0 else 0.0
    return {
        "feature": feature,
        "type": "categorical",
        "information_gain": info_gain,
        "gain_ratio": gain_ratio,
        "gini_gain": gini_gain,
        "branches": frame[feature].nunique(),
    }

pd.DataFrame([categorical_split_score(data, "outlook")])

In [ ]:
def best_threshold_score(frame, feature, target="play"):
    order = frame.sort_values(feature)
    x = order[feature].to_numpy()
    y = order[target].to_numpy()
    candidates = []
    for i in range(1, len(order)):
        if y[i - 1] != y[i] and x[i - 1] != x[i]:
            candidates.append((x[i - 1] + x[i]) / 2)
    rows = []
    parent_entropy = entropy(y)
    parent_gini = gini(y)
    for threshold in candidates:
        left = order.loc[order[feature] <= threshold, target].to_numpy()
        right = order.loc[order[feature] > threshold, target].to_numpy()
        rows.append({
            "feature": feature,
            "threshold": threshold,
            "information_gain": parent_entropy - weighted_impurity([left, right], entropy),
            "gini_gain": parent_gini - weighted_impurity([left, right], gini),
            "left_n": len(left),
            "right_n": len(right),
        })
    return pd.DataFrame(rows).sort_values("information_gain", ascending=False)

threshold_tables = {feature: best_threshold_score(data, feature) for feature in ["humidity", "wind", "temperature"]}
pd.concat([table.head(1) for table in threshold_tables.values()], ignore_index=True)

In [ ]:
all_scores = [categorical_split_score(data, "outlook")]
for feature, table in threshold_tables.items():
    best = table.head(1).iloc[0].to_dict()
    best.update({"feature": feature, "type": "continuous", "gain_ratio": np.nan, "branches": 2})
    all_scores.append(best)

score_table = pd.DataFrame(all_scores)[["feature", "type", "threshold", "information_gain", "gain_ratio", "gini_gain", "branches"]]
score_table.sort_values("information_gain", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, feature in zip(axes, ["humidity", "wind", "temperature"]):
    table = threshold_tables[feature]
    ax.plot(table["threshold"], table["information_gain"], label="information gain")
    ax.plot(table["threshold"], table["gini_gain"], label="Gini gain", alpha=0.75)
    ax.set_title(feature)
    ax.set_xlabel("candidate threshold")
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel("split improvement")
axes[0].legend()
fig.suptitle("Continuous split search scans candidate thresholds")
fig.tight_layout()

## Takeaway

A tree algorithm is defined as much by its split score as by its tree structure. ID3-style information gain, C4.5-style gain ratio, CART-style Gini or squared-error reduction, and statistical-test criteria can choose different roots on the same data.